In [ ]:
"""
Real-world Agentic AI demonstration:
Vehicle Telemetry Monitoring System

Concepts demonstrated:
1. Main Agent
2. Static Subagents
3. Dynamic Subagents
4. Task delegation
5. Conditional specialist creation
6. Result aggregation

No external libraries are required.

Run:
    python telemetry_agent_system.py
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional, Type
import random


# ============================================================
# Domain models
# ============================================================

class Severity(Enum):
    NORMAL = "NORMAL"
    WARNING = "WARNING"
    CRITICAL = "CRITICAL"


@dataclass
class TelemetryReading:
    vehicle_id: str
    timestamp: datetime

    speed_kmph: float
    engine_temperature_c: float
    battery_voltage_v: float
    oil_pressure_psi: float
    brake_pad_percent: float
    tire_pressure_psi: float
    fuel_percent: float


@dataclass
class Finding:
    source: str
    component: str
    severity: Severity
    message: str
    recommendation: str


@dataclass
class AgentResult:
    agent_name: str
    agent_category: str
    findings: List[Finding] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


# ============================================================
# Base agent
# ============================================================

class BaseAgent(ABC):
    """Base class shared by all static and dynamic subagents."""

    def __init__(self, name: str, responsibility: str) -> None:
        self.name = name
        self.responsibility = responsibility

    @abstractmethod
    def execute(self, context: Dict[str, Any]) -> AgentResult:
        """Perform the assigned responsibility."""
        raise NotImplementedError


# ============================================================
# Static subagent 1: Data validation
#
# This subagent is created when the application starts.
# It validates every telemetry reading.
# ============================================================

class DataValidationSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Data Validation Subagent",
            responsibility="Validate telemetry values and required fields",
        )

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        reading: TelemetryReading = context["reading"]
        findings: List[Finding] = []

        validation_rules = {
            "speed_kmph": (0, 300),
            "engine_temperature_c": (-40, 180),
            "battery_voltage_v": (0, 30),
            "oil_pressure_psi": (0, 150),
            "brake_pad_percent": (0, 100),
            "tire_pressure_psi": (0, 100),
            "fuel_percent": (0, 100),
        }

        for field_name, (minimum, maximum) in validation_rules.items():
            value = getattr(reading, field_name)

            if not minimum <= value <= maximum:
                findings.append(
                    Finding(
                        source=self.name,
                        component="Telemetry data",
                        severity=Severity.CRITICAL,
                        message=(
                            f"Invalid {field_name}: {value}. "
                            f"Expected range is {minimum}–{maximum}."
                        ),
                        recommendation="Inspect the sensor or incoming data feed.",
                    )
                )

        if not findings:
            findings.append(
                Finding(
                    source=self.name,
                    component="Telemetry data",
                    severity=Severity.NORMAL,
                    message="All telemetry values passed validation.",
                    recommendation="No action required.",
                )
            )

        return AgentResult(
            agent_name=self.name,
            agent_category="Static Subagent",
            findings=findings,
        )


# ============================================================
# Static subagent 2: General health analysis
#
# This subagent performs a first-level assessment and identifies
# which specialist agents may be required.
# ============================================================

class VehicleHealthSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Vehicle Health Subagent",
            responsibility="Perform first-level vehicle health assessment",
        )

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        reading: TelemetryReading = context["reading"]

        findings: List[Finding] = []
        specialist_requests: List[str] = []

        # Engine temperature analysis
        if reading.engine_temperature_c >= 115:
            findings.append(
                Finding(
                    source=self.name,
                    component="Engine cooling",
                    severity=Severity.CRITICAL,
                    message=(
                        f"Engine temperature is critically high: "
                        f"{reading.engine_temperature_c:.1f}°C."
                    ),
                    recommendation="Request detailed thermal diagnosis.",
                )
            )
            specialist_requests.append("thermal")

        elif reading.engine_temperature_c >= 105:
            findings.append(
                Finding(
                    source=self.name,
                    component="Engine cooling",
                    severity=Severity.WARNING,
                    message=(
                        f"Engine temperature is elevated: "
                        f"{reading.engine_temperature_c:.1f}°C."
                    ),
                    recommendation="Monitor the cooling system.",
                )
            )
            specialist_requests.append("thermal")

        # Battery analysis
        if reading.battery_voltage_v < 11.8:
            findings.append(
                Finding(
                    source=self.name,
                    component="Battery",
                    severity=Severity.CRITICAL,
                    message=(
                        f"Battery voltage is critically low: "
                        f"{reading.battery_voltage_v:.1f} V."
                    ),
                    recommendation="Request battery specialist analysis.",
                )
            )
            specialist_requests.append("battery")

        elif reading.battery_voltage_v < 12.2:
            findings.append(
                Finding(
                    source=self.name,
                    component="Battery",
                    severity=Severity.WARNING,
                    message=(
                        f"Battery voltage is below the preferred level: "
                        f"{reading.battery_voltage_v:.1f} V."
                    ),
                    recommendation="Inspect the charging system.",
                )
            )
            specialist_requests.append("battery")

        # Brake analysis
        if reading.brake_pad_percent <= 10:
            findings.append(
                Finding(
                    source=self.name,
                    component="Braking system",
                    severity=Severity.CRITICAL,
                    message=(
                        f"Brake pad life is critically low: "
                        f"{reading.brake_pad_percent:.1f}%."
                    ),
                    recommendation="Request immediate brake diagnosis.",
                )
            )
            specialist_requests.append("brake")

        elif reading.brake_pad_percent <= 20:
            findings.append(
                Finding(
                    source=self.name,
                    component="Braking system",
                    severity=Severity.WARNING,
                    message=(
                        f"Brake pad life is low: "
                        f"{reading.brake_pad_percent:.1f}%."
                    ),
                    recommendation="Schedule brake maintenance.",
                )
            )
            specialist_requests.append("brake")

        # Oil pressure
        if reading.oil_pressure_psi < 20:
            findings.append(
                Finding(
                    source=self.name,
                    component="Lubrication system",
                    severity=Severity.CRITICAL,
                    message=(
                        f"Oil pressure is dangerously low: "
                        f"{reading.oil_pressure_psi:.1f} PSI."
                    ),
                    recommendation="Stop the engine and inspect the oil system.",
                )
            )
            specialist_requests.append("engine")

        # Tire pressure
        if reading.tire_pressure_psi < 28:
            findings.append(
                Finding(
                    source=self.name,
                    component="Tires",
                    severity=Severity.WARNING,
                    message=(
                        f"Tire pressure is low: "
                        f"{reading.tire_pressure_psi:.1f} PSI."
                    ),
                    recommendation="Inflate and inspect the tires.",
                )
            )
            specialist_requests.append("tire")

        if not findings:
            findings.append(
                Finding(
                    source=self.name,
                    component="Overall vehicle",
                    severity=Severity.NORMAL,
                    message="No vehicle-health problems were detected.",
                    recommendation="Continue normal operation.",
                )
            )

        return AgentResult(
            agent_name=self.name,
            agent_category="Static Subagent",
            findings=findings,
            metadata={
                "specialist_requests": list(dict.fromkeys(specialist_requests))
            },
        )


# ============================================================
# Dynamic subagents
#
# These objects do not exist permanently.
# The main agent creates them only when a relevant fault occurs.
# ============================================================

class BatteryDynamicSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Dynamic Battery Specialist",
            responsibility="Diagnose battery and charging-system problems",
        )
        print(f"      [CREATED] {self.name}")

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        reading: TelemetryReading = context["reading"]
        findings: List[Finding] = []

        voltage = reading.battery_voltage_v

        if voltage < 11.5:
            message = (
                "The battery may be deeply discharged, damaged, "
                "or disconnected from the charging system."
            )
            action = (
                "Stop nonessential electrical loads. Test the battery, "
                "alternator and terminal connections immediately."
            )
            severity = Severity.CRITICAL
        else:
            message = (
                "Battery voltage suggests partial discharge or "
                "weak charging performance."
            )
            action = (
                "Perform a load test and inspect alternator output "
                "and cable connections."
            )
            severity = Severity.WARNING

        findings.append(
            Finding(
                source=self.name,
                component="Battery and charging system",
                severity=severity,
                message=message,
                recommendation=action,
            )
        )

        return AgentResult(
            agent_name=self.name,
            agent_category="Dynamic Subagent",
            findings=findings,
        )


class ThermalDynamicSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Dynamic Thermal Specialist",
            responsibility="Diagnose engine overheating conditions",
        )
        print(f"      [CREATED] {self.name}")

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        reading: TelemetryReading = context["reading"]

        if reading.engine_temperature_c >= 115:
            finding = Finding(
                source=self.name,
                component="Cooling system",
                severity=Severity.CRITICAL,
                message=(
                    "Probable overheating condition. Possible causes include "
                    "low coolant, fan failure, pump failure or thermostat fault."
                ),
                recommendation=(
                    "Reduce engine load and stop safely. Inspect coolant level, "
                    "radiator fan, water pump and thermostat."
                ),
            )
        else:
            finding = Finding(
                source=self.name,
                component="Cooling system",
                severity=Severity.WARNING,
                message=(
                    "Thermal trend is abnormal but has not yet reached "
                    "the critical shutdown threshold."
                ),
                recommendation=(
                    "Monitor temperature and inspect the cooling system "
                    "at the next service opportunity."
                ),
            )

        return AgentResult(
            agent_name=self.name,
            agent_category="Dynamic Subagent",
            findings=[finding],
        )


class BrakeDynamicSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Dynamic Brake Specialist",
            responsibility="Diagnose brake wear and safety risks",
        )
        print(f"      [CREATED] {self.name}")

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        reading: TelemetryReading = context["reading"]

        if reading.brake_pad_percent <= 10:
            severity = Severity.CRITICAL
            message = "Brake pads have reached a safety-critical wear level."
            recommendation = (
                "Restrict vehicle operation and replace the brake pads "
                "before normal use."
            )
        else:
            severity = Severity.WARNING
            message = "Brake pads are approaching their replacement threshold."
            recommendation = (
                "Schedule brake-pad inspection and replacement."
            )

        return AgentResult(
            agent_name=self.name,
            agent_category="Dynamic Subagent",
            findings=[
                Finding(
                    source=self.name,
                    component="Braking system",
                    severity=severity,
                    message=message,
                    recommendation=recommendation,
                )
            ],
        )


class EngineDynamicSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Dynamic Engine Specialist",
            responsibility="Diagnose engine lubrication problems",
        )
        print(f"      [CREATED] {self.name}")

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        return AgentResult(
            agent_name=self.name,
            agent_category="Dynamic Subagent",
            findings=[
                Finding(
                    source=self.name,
                    component="Engine lubrication",
                    severity=Severity.CRITICAL,
                    message=(
                        "Low oil pressure may indicate low oil level, "
                        "pump failure, leakage or internal engine wear."
                    ),
                    recommendation=(
                        "Shut down the engine and inspect oil level, "
                        "pressure sensor, filter, pump and possible leakage."
                    ),
                )
            ],
        )


class TireDynamicSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Dynamic Tire Specialist",
            responsibility="Diagnose tire-pressure problems",
        )
        print(f"      [CREATED] {self.name}")

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        reading: TelemetryReading = context["reading"]

        return AgentResult(
            agent_name=self.name,
            agent_category="Dynamic Subagent",
            findings=[
                Finding(
                    source=self.name,
                    component="Tires",
                    severity=Severity.WARNING,
                    message=(
                        f"Tire pressure of {reading.tire_pressure_psi:.1f} PSI "
                        "may increase tire wear and reduce vehicle stability."
                    ),
                    recommendation=(
                        "Inspect for punctures and inflate the tire to "
                        "the manufacturer-recommended pressure."
                    ),
                )
            ],
        )


# ============================================================
# Factory for creating dynamic specialists
# ============================================================

class DynamicSpecialistFactory:

    _registry: Dict[str, Type[BaseAgent]] = {
        "battery": BatteryDynamicSubAgent,
        "thermal": ThermalDynamicSubAgent,
        "brake": BrakeDynamicSubAgent,
        "engine": EngineDynamicSubAgent,
        "tire": TireDynamicSubAgent,
    }

    @classmethod
    def create(cls, specialist_type: str) -> BaseAgent:
        agent_class = cls._registry.get(specialist_type)

        if agent_class is None:
            raise ValueError(
                f"No dynamic specialist registered for '{specialist_type}'."
            )

        return agent_class()


# ============================================================
# Static report-generation subagent
# ============================================================

class ReportGenerationSubAgent(BaseAgent):

    def __init__(self) -> None:
        super().__init__(
            name="Report Generation Subagent",
            responsibility="Create the final vehicle health report",
        )

    def execute(self, context: Dict[str, Any]) -> AgentResult:
        findings: List[Finding] = context["findings"]

        severity_rank = {
            Severity.NORMAL: 0,
            Severity.WARNING: 1,
            Severity.CRITICAL: 2,
        }

        highest_severity = max(
            findings,
            key=lambda finding: severity_rank[finding.severity],
        ).severity

        return AgentResult(
            agent_name=self.name,
            agent_category="Static Subagent",
            metadata={
                "overall_status": highest_severity.value,
                "finding_count": len(findings),
            },
        )


# ============================================================
# Main agent
#
# Owns the complete goal:
# "Monitor the vehicle and produce an actionable health report."
# ============================================================

class TelemetryMainAgent:

    def __init__(self) -> None:
        self.name = "Vehicle Telemetry Main Agent"

        # These static subagents are created once and reused.
        self.validation_agent = DataValidationSubAgent()
        self.health_agent = VehicleHealthSubAgent()
        self.report_agent = ReportGenerationSubAgent()

        print("=" * 75)
        print(f"{self.name} initialized")
        print("Static subagents are permanently available:")
        print("  1. Data Validation Subagent")
        print("  2. Vehicle Health Subagent")
        print("  3. Report Generation Subagent")
        print("=" * 75)

    def monitor(self, reading: TelemetryReading) -> str:
        context = {"reading": reading}
        all_findings: List[Finding] = []
        execution_log: List[str] = []

        print(f"\nMonitoring vehicle: {reading.vehicle_id}")
        print(f"Timestamp: {reading.timestamp.isoformat()}")

        # ----------------------------------------------------
        # Step 1: Static validation subagent
        # ----------------------------------------------------

        print("\n[MAIN AGENT] Delegating data validation...")

        validation_result = self.validation_agent.execute(context)
        all_findings.extend(validation_result.findings)
        execution_log.append(validation_result.agent_name)

        invalid_data = any(
            finding.severity == Severity.CRITICAL
            for finding in validation_result.findings
        )

        if invalid_data:
            return self._build_report(
                reading=reading,
                findings=all_findings,
                execution_log=execution_log,
            )

        # ----------------------------------------------------
        # Step 2: Static health-analysis subagent
        # ----------------------------------------------------

        print("[MAIN AGENT] Delegating first-level health analysis...")

        health_result = self.health_agent.execute(context)
        all_findings.extend(health_result.findings)
        execution_log.append(health_result.agent_name)

        specialist_requests = health_result.metadata.get(
            "specialist_requests",
            [],
        )

        # ----------------------------------------------------
        # Step 3: Create required dynamic subagents
        # ----------------------------------------------------

        if specialist_requests:
            print(
                "[MAIN AGENT] Problems detected. "
                "Creating dynamic specialists:"
            )

            for specialist_type in specialist_requests:
                specialist = DynamicSpecialistFactory.create(
                    specialist_type
                )

                specialist_result = specialist.execute(context)
                all_findings.extend(specialist_result.findings)
                execution_log.append(specialist_result.agent_name)

                print(f"      [RELEASED] {specialist.name}")

                # It is not retained by the main agent.
                del specialist

        else:
            print(
                "[MAIN AGENT] No specialist diagnosis is required. "
                "No dynamic subagent was created."
            )

        # ----------------------------------------------------
        # Step 4: Static reporting subagent
        # ----------------------------------------------------

        print("[MAIN AGENT] Delegating final report generation...")

        report_result = self.report_agent.execute(
            {"findings": all_findings}
        )
        execution_log.append(report_result.agent_name)

        return self._build_report(
            reading=reading,
            findings=all_findings,
            execution_log=execution_log,
            overall_status=report_result.metadata["overall_status"],
        )

    @staticmethod
    def _build_report(
        reading: TelemetryReading,
        findings: List[Finding],
        execution_log: List[str],
        overall_status: Optional[str] = None,
    ) -> str:

        if overall_status is None:
            overall_status = "CRITICAL"

        lines = [
            "",
            "=" * 75,
            "VEHICLE TELEMETRY HEALTH REPORT",
            "=" * 75,
            f"Vehicle          : {reading.vehicle_id}",
            f"Timestamp        : {reading.timestamp}",
            f"Overall status   : {overall_status}",
            "",
            "Telemetry values",
            "-" * 75,
            f"Speed            : {reading.speed_kmph:.1f} km/h",
            f"Engine temperature: {reading.engine_temperature_c:.1f} °C",
            f"Battery voltage  : {reading.battery_voltage_v:.1f} V",
            f"Oil pressure     : {reading.oil_pressure_psi:.1f} PSI",
            f"Brake pad life   : {reading.brake_pad_percent:.1f}%",
            f"Tire pressure    : {reading.tire_pressure_psi:.1f} PSI",
            f"Fuel level       : {reading.fuel_percent:.1f}%",
            "",
            "Findings",
            "-" * 75,
        ]

        for number, finding in enumerate(findings, start=1):
            lines.extend(
                [
                    f"{number}. [{finding.severity.value}] "
                    f"{finding.component}",
                    f"   Agent          : {finding.source}",
                    f"   Observation    : {finding.message}",
                    f"   Recommendation : {finding.recommendation}",
                    "",
                ]
            )

        lines.extend(
            [
                "Agent execution trace",
                "-" * 75,
            ]
        )

        for number, agent_name in enumerate(execution_log, start=1):
            lines.append(f"{number}. {agent_name}")

        lines.extend(
            [
                "",
                "Concept demonstrated",
                "-" * 75,
                "Main Agent:",
                "  Owned and coordinated the complete monitoring workflow.",
                "",
                "Static Subagents:",
                "  Validation, general health analysis and reporting were",
                "  permanently available and executed for every reading.",
                "",
                "Dynamic Subagents:",
                "  Specialist agents were created only for detected faults",
                "  and released after returning their diagnoses.",
                "=" * 75,
            ]
        )

        return "\n".join(lines)


# ============================================================
# Telemetry simulator
# ============================================================

def generate_normal_reading(vehicle_id: str) -> TelemetryReading:
    return TelemetryReading(
        vehicle_id=vehicle_id,
        timestamp=datetime.now(),
        speed_kmph=random.uniform(40, 90),
        engine_temperature_c=random.uniform(85, 98),
        battery_voltage_v=random.uniform(12.4, 13.8),
        oil_pressure_psi=random.uniform(35, 55),
        brake_pad_percent=random.uniform(55, 90),
        tire_pressure_psi=random.uniform(31, 35),
        fuel_percent=random.uniform(30, 90),
    )


def generate_fault_reading(vehicle_id: str) -> TelemetryReading:
    return TelemetryReading(
        vehicle_id=vehicle_id,
        timestamp=datetime.now(),
        speed_kmph=72,
        engine_temperature_c=119,
        battery_voltage_v=11.4,
        oil_pressure_psi=16,
        brake_pad_percent=8,
        tire_pressure_psi=25,
        fuel_percent=32,
    )


# ============================================================
# Application entry point
# ============================================================

def main() -> None:
    main_agent = TelemetryMainAgent()

    print("\n\nDEMO 1: HEALTHY VEHICLE")
    normal_reading = generate_normal_reading("TRUCK-101")
    normal_report = main_agent.monitor(normal_reading)
    print(normal_report)

    print("\n\nDEMO 2: VEHICLE WITH MULTIPLE FAULTS")
    faulty_reading = generate_fault_reading("TRUCK-202")
    faulty_report = main_agent.monitor(faulty_reading)
    print(faulty_report)


if __name__ == "__main__":
    main()